In [ ]:
import csv
import random
from datetime import datetime, timedelta

In [ ]:
# Configurações do dataset
NUM_ROWS = 25
ANOMALY_PERCENTAGE = 0.30
FILENAME = "telemetry_anomalies_dataset.csv"

In [ ]:
# Calcula quantas linhas terão anomalias (3% de 500 = 15)
num_anomalies = int(NUM_ROWS * ANOMALY_PERCENTAGE)
# Sorteia quais índices das 500 linhas serão os anômalos
anomaly_indices = set(random.sample(range(NUM_ROWS), num_anomalies))

In [ ]:
# Tipos de anomalias possíveis
ANOMALY_TYPES = [
    'high_temp', 
    'low_battery', 
    'pressure_low', 
    'pressure_high', 
    'struct_fail', 
    'crit_mod_fail', 
    'telemetry_fail'
]

In [ ]:
# Função para verificar condições seguras (Decisão READY/ABORT)
def is_safe(internal_temp, external_temp, battery_v, battery_soc, pressure, struct, crit, telem):
    if not (20 <= internal_temp <= 30): return False
    if external_temp < 0: return False
    if battery_v < 48: return False
    if battery_soc < 80: return False
    if not (95 <= pressure <= 135): return False  # Limite seguro de pressão
    if struct != 1: return False
    if crit != 1: return False
    if telem != 1: return False
    return True

In [ ]:
def generate_dataset():
    start_time = datetime.utcnow()
    
    with open(FILENAME, mode='w', newline='') as file:
        writer = csv.writer(file)
        
        # Cabeçalho com a nova coluna
        writer.writerow([
            "timestamp", "internal_temp_c", "external_temp_c", "battery_voltage_v",
            "battery_current_a", "battery_soc_percent", "battery_capacity_ah",
            "energy_available_kwh", "power_load_kw", "energy_loss_percent",
            "tank_pressure_bar", "structural_integrity", "critical_modules_status",
            "telemetry_link_status", "estimated_autonomy_min", "launch_decision",
            "anomalia_inserida"
        ])
        
        for i in range(NUM_ROWS):
            # 1. Geração de Dados Normais
            timestamp = (start_time + timedelta(seconds=i)).strftime('%Y-%m-%dT%H:%M:%SZ')
            internal_temp = round(random.uniform(18, 35), 1)
            external_temp = round(random.uniform(-5, 30), 1)
            battery_v = round(random.uniform(46, 52), 1)
            battery_current = round(random.uniform(20, 120), 1)
            battery_soc = round(random.uniform(60, 100), 1)
            battery_cap = round(random.uniform(80, 120), 1)
            power_load = round(random.uniform(5, 25), 1)
            energy_loss = round(random.uniform(2, 8), 1)
            tank_pressure = round(random.uniform(95, 145), 1)
            autonomy = int(random.uniform(45, 180))
            
            struct_int = 1
            crit_mod = 1
            telemetry = 1
            
            is_anomalous = "nao"
            
            # 2. Inserção de Anomalia (se o índice foi sorteado)
            if i in anomaly_indices:
                is_anomalous = "sim"
                anomaly_to_inject = random.choice(ANOMALY_TYPES)
                
                if anomaly_to_inject == 'high_temp':
                    internal_temp = round(random.uniform(45, 70), 1)
                elif anomaly_to_inject == 'low_battery':
                    battery_soc = round(random.uniform(5, 25), 1)
                elif anomaly_to_inject == 'pressure_low':
                    tank_pressure = round(random.uniform(40, 80), 1)
                elif anomaly_to_inject == 'pressure_high':
                    tank_pressure = round(random.uniform(160, 220), 1)
                elif anomaly_to_inject == 'struct_fail':
                    struct_int = 0
                elif anomaly_to_inject == 'crit_mod_fail':
                    crit_mod = 0
                elif anomaly_to_inject == 'telemetry_fail':
                    telemetry = 0

            # 3. Recálculo Pós-Anomalia
            # Energia Disponível
            energy_kwh = (battery_v * battery_cap * (battery_soc / 100)) / 1000
            energy_kwh = round(energy_kwh, 3)
            
            # Decisão de Lançamento
            if is_safe(internal_temp, external_temp, battery_v, battery_soc, tank_pressure, struct_int, crit_mod, telemetry):
                decision = "READY"
            else:
                decision = "ABORT"
                
            writer.writerow([
                timestamp, internal_temp, external_temp, battery_v, battery_current,
                battery_soc, battery_cap, energy_kwh, power_load, energy_loss,
                tank_pressure, struct_int, crit_mod, telemetry, autonomy, decision,
                is_anomalous
            ])
            
    print(f"Sucesso: Dataset '{FILENAME}' gerado com {NUM_ROWS} linhas!")
    print(f"Foram inseridas {num_anomalies} anomalias controladas (30% do total).")

In [ ]:
if __name__ == "__main__":
    generate_dataset()